In [13]:
import random
from datasets import load_dataset

# Load the dataset (skip if already loaded)
ds_math = load_dataset("open-r1/OpenThoughts-114k-math")

# Sort all examples by token count
all_examples = list(ds_math['train'])
all_examples.sort(key=lambda x: x['generated_token_count'])
n = len(all_examples)

# Difficulty boundaries (from our Day 1 analysis)
p25 = all_examples[n // 4]['generated_token_count']  # ~2820
p75 = all_examples[3 * n // 4]['generated_token_count']  # ~8648

easy = [ex for ex in all_examples if ex['generated_token_count'] < p25]
medium = [ex for ex in all_examples if p25 <= ex['generated_token_count'] < p75]
hard = [ex for ex in all_examples if ex['generated_token_count'] >= p75]

print(f"Easy: {len(easy)} | Medium: {len(medium)} | Hard: {len(hard)}")

# Sample 2000 from each
random.seed(42)
easy_sample = random.sample(easy, 2000)
medium_sample = random.sample(medium, 2000)
hard_sample = random.sample(hard, 2000)

print(f"Sampled: {len(easy_sample)} easy + {len(medium_sample)} medium + {len(hard_sample)} hard")

# Parse each example into clean format
def parse_example(ex, difficulty):
    response = ex['conversations'][1]['value']
    
    # Extract thinking
    if '<|end_of_thought|>' in response:
        thinking = response.split('<|end_of_thought|>')[0].replace('<|begin_of_thought|>', '').strip()
    else:
        thinking = ""
    
    # Extract answer
    if '<|begin_of_solution|>' in response and '<|end_of_solution|>' in response:
        answer = response.split('<|begin_of_solution|>')[1].split('<|end_of_solution|>')[0].strip()
    elif '<|end_of_thought|>' in response:
        answer = response.split('<|end_of_thought|>')[1].strip()
    else:
        answer = response
    
    return {
        'problem': ex['problem'],
        'source': ex['source'],
        'ground_truth': ex['solution'],
        'thinking': thinking,
        'answer': answer,
        'difficulty': difficulty,
        'original_token_count': ex['generated_token_count'],
        'thinking_word_count': len(thinking.split())
    }

# Parse all sampled examples
parsed = []
for ex in easy_sample:
    parsed.append(parse_example(ex, 'easy'))
for ex in medium_sample:
    parsed.append(parse_example(ex, 'medium'))
for ex in hard_sample:
    parsed.append(parse_example(ex, 'hard'))

print(f"\nTotal parsed: {len(parsed)}")

# Verify parsing worked
for diff in ['easy', 'medium', 'hard']:
    subset = [p for p in parsed if p['difficulty'] == diff]
    avg_words = sum(p['thinking_word_count'] for p in subset) // len(subset)
    print(f"{diff}: {len(subset)} examples, avg thinking: {avg_words} words")

Easy: 22272 | Medium: 44567 | Hard: 22281
Sampled: 2000 easy + 2000 medium + 2000 hard

Total parsed: 6000
easy: 2000 examples, avg thinking: 964 words
medium: 2000 examples, avg thinking: 2789 words
hard: 2000 examples, avg thinking: 7640 words


In [14]:
import json

with open('parsed_6000.json', 'w') as f:
    json.dump(parsed, f)

print(f"Saved {len(parsed)} examples to parsed_6000.json")

Saved 6000 examples to parsed_6000.json


In [2]:
# Look at one example from each difficulty
for diff in ['easy', 'medium', 'hard']:
    ex = [p for p in parsed if p['difficulty'] == diff][0]
    print(f"\n{'='*60}")
    print(f"DIFFICULTY: {diff.upper()}")
    print(f"Source: {ex['source']} | Thinking: {ex['thinking_word_count']} words")
    print(f"\nPROBLEM:\n{ex['problem'][:300]}")
    print(f"\nTHINKING (first 400 chars):\n{ex['thinking'][:400]}")
    print(f"\nANSWER (first 300 chars):\n{ex['answer'][:300]}")


DIFFICULTY: EASY
Source: olympiads | Thinking: 1385 words

PROBLEM:
Given a convex polygon \( A_{n+1} \) with \( n+1 \) sides (where \( n \geq 2 \)), partition the polygon into \( n-1 \) triangles using \( n-2 \) non-intersecting diagonals (referred to as a triangulation of \( A_{n+1} \)). How many different triangulation methods are there?

THINKING (first 400 chars):
Okay, so I need to figure out how many different triangulation methods there are for a convex polygon with n+1 sides. The problem says that when you triangulate it, you end up with n-1 triangles using n-2 non-intersecting diagonals. Hmm, let me start by recalling what a triangulation is. 

A triangulation of a convex polygon is a division of the polygon into triangles by drawing diagonals that don

ANSWER (first 300 chars):
To determine the number of different triangulation methods for a convex polygon \( A_{n+1} \) with \( n+1 \) sides, we recognize that this problem is related to the Catalan numbers. 

A triangulation

In [5]:
import os
from dotenv import load_dotenv
import google.genai as genai

load_dotenv()

client=genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash-lite", 
    contents="What is 2+2? Answer in one word."
)

print(response.text)

Four


In [6]:
import json

with open('parsed_6000.json', 'r') as f:
    parsed = json.load(f)

print(f"Loaded {len(parsed)} examples")

Loaded 6000 examples


In [7]:
COMPRESS_PROMPT = """You are a reasoning trace compressor.

Below is a LONG reasoning trace from a math problem. Your job is to rewrite it as a SHORT, 
direct solution that keeps ONLY the essential logical steps needed to reach the answer.

REMOVE:
- Self-doubt ("let me think", "hmm", "wait")
- Redundant verification ("let me check", "let me verify")
- Repeated explanations of the same idea
- Dead ends and backtracking
- Obvious steps that add no value

KEEP:
- The core logical chain from problem to answer
- Key equations and calculations
- Critical insights that lead to the solution

RULES:
- The compressed trace MUST lead to the same final answer
- Target length: {target_words} words maximum
- Write in first person, natural reasoning style
- Do NOT include the final answer — only the thinking steps

ORIGINAL TRACE:
{trace}

Write the compressed reasoning trace:"""

# Test on 3 easy examples
easy_examples = [p for p in parsed if p['difficulty'] == 'easy']

for i in range(3):
    ex = easy_examples[i]
    prompt = COMPRESS_PROMPT.format(
        target_words=150,
        trace=ex['thinking']
    )
    
    client=genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))
    
    # response = model.generate_content(prompt)
    response = client.models.generate_content(
    model="gemini-2.5-flash-lite", 
    contents=prompt
)

    compressed = response.text.strip()
    
    print(f"\n{'='*60}")
    print(f"EXAMPLE {i}")
    print(f"Problem: {ex['problem'][:150]}...")
    print(f"\nORIGINAL: {ex['thinking_word_count']} words")
    print(f"COMPRESSED: {len(compressed.split())} words")
    print(f"Compression ratio: {ex['thinking_word_count'] / max(len(compressed.split()), 1):.1f}x")
    print(f"\nCOMPRESSED TRACE:")
    print(compressed[:500])


EXAMPLE 0
Problem: Given a convex polygon \( A_{n+1} \) with \( n+1 \) sides (where \( n \geq 2 \)), partition the polygon into \( n-1 \) triangles using \( n-2 \) non-i...

ORIGINAL: 1385 words
COMPRESSED: 155 words
Compression ratio: 8.9x

COMPRESSED TRACE:
I need to find the number of triangulations for a convex polygon with $n+1$ sides.
I recall that the number of triangulations for a convex polygon with $m$ sides is given by the $(m-2)$-th Catalan number, $C_{m-2}$.
The formula for the $k$-th Catalan number is $C_k = \frac{1}{k+1}\binom{2k}{k}$.

Since the polygon has $n+1$ sides, we set $m = n+1$.
Therefore, the number of triangulations is $C_{(n+1)-2} = C_{n-1}$.

Substituting $k = n-1$ into the Catalan number formula:
$C_{n-1} = \frac{1}{(n-1

EXAMPLE 1
Problem: The locus of the complex number \( z \) that satisfies \( 2|z-2+\mathrm{i}| + 2|z+3-\mathrm{i}| = 12 \) is:
A. Circle  
B. Ellipse  
C. Hyperbola  
D....

ORIGINAL: 701 words
COMPRESSED: 155 words
Compression ratio: 4.

In [9]:
easy_examples = [p for p in parsed if p['difficulty'] == 'easy']

for i in range(3):
    ex = easy_examples[i]
    prompt = COMPRESS_PROMPT.format(
        target_words=150,
        trace=ex['thinking']
    )
    response = client.models.generate_content(
    model="gemini-2.5-flash-lite", 
    contents=prompt)
    
    compressed = response.text.strip()
    
    original_words = ex['thinking_word_count']
    compressed_words = len(compressed.split())
    
    print(f"\n{'='*60}")
    print(f"EXAMPLE {i}")
    print(f"Problem: {ex['problem'][:150]}")
    print(f"Original: {original_words} words → Compressed: {compressed_words} words ({original_words/max(compressed_words,1):.1f}x)")
    print(f"\nCOMPRESSED:")
    print(compressed)
    print(f"\nEXPECTED ANSWER (first 200 chars):")
    print(ex['answer'][:200])


EXAMPLE 0
Problem: Given a convex polygon \( A_{n+1} \) with \( n+1 \) sides (where \( n \geq 2 \)), partition the polygon into \( n-1 \) triangles using \( n-2 \) non-i
Original: 1385 words → Compressed: 120 words (11.5x)

COMPRESSED:
I need to find the number of triangulations for a convex polygon with n+1 sides. I recall that the number of triangulations for a convex polygon with $m$ sides is given by the $(m-2)$th Catalan number, $C_{m-2}$. The formula for the $k$th Catalan number is $C_k = \frac{1}{k+1} \binom{2k}{k}$.

In this problem, the polygon has $n+1$ sides. So, $m = n+1$.
The number of triangulations will be $C_{(n+1)-2} = C_{n-1}$.

Substituting $k = n-1$ into the Catalan number formula:
$C_{n-1} = \frac{1}{(n-1)+1} \binom{2(n-1)}{n-1} = \frac{1}{n} \binom{2n-2}{n-1}$.

This can also be written as $\frac{(2n-2)!}{n!(n-1)!}$. This result is confirmed by checking small cases: a triangle (3 sides, $n=2$) has 1 triangulation ($C_1=1$), and a quadrilateral (4 sides, $n=3$) ha

In [10]:
for diff in ['easy', 'medium', 'hard']:
    subset = [p for p in parsed if p['difficulty'] == diff]
    ex = subset[0]
    print(f"\n{'='*40}")
    print(f"{diff.upper()} — answer field ({len(ex['answer'])} chars):")
    print(ex['answer'][:300])


EASY — answer field (748 chars):
To determine the number of different triangulation methods for a convex polygon \( A_{n+1} \) with \( n+1 \) sides, we recognize that this problem is related to the Catalan numbers. 

A triangulation of a convex polygon with \( m \) sides is given by the \((m-2)\)th Catalan number. For a polygon wit

MEDIUM — answer field (1380 chars):
To find the maximum possible value of \( PA + PB + PC + PD \) where \( P \) is a point inside the unit square \( ABCD \), we start by considering the coordinates of the square. Let \( A = (0,0) \), \( B = (1,0) \), \( C = (1,1) \), and \( D = (0,1) \). For any point \( P \) inside the square with co

HARD — answer field (1816 chars):
To prove that all numbers in the set \( M = \{1, 2, \ldots, n-1\} \) are the same color under the given conditions, we analyze the connectivity imposed by the coloring rules and the coprimality of \( k \) and \( n \).

### Key Steps and Reasoning:

1. **Complement Pairing (Condition a):**  
